## df = spark.read.format("delta").table("schema.tabela")

Gerando um dataframe dos delta lake no container bronze do Azure Data Lake Storage

In [0]:
df_agent_policies  = spark.read.format("delta").table("silver.agent_policies")
df_agents          = spark.read.format("delta").table("silver.agents")
df_claims          = spark.read.format("delta").table("silver.claims")
df_customers       = spark.read.format("delta").table("silver.customers")
df_insurance_types = spark.read.format("delta").table("silver.insurance_types")
df_payments        = spark.read.format("delta").table("silver.payments")
df_policies        = spark.read.format("delta").table("silver.policies")

### Adicionando metadados de data e hora de processamento e nome do arquivo de origem

In [0]:
%sql
DROP TABLE IF EXISTS gold.dim_customers;

In [0]:
%sql
CREATE TABLE gold.dim_customers (
    sk_customer BIGINT GENERATED BY DEFAULT AS IDENTITY,
    customer_id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    date_of_birth DATE
) USING DELTA;

In [0]:
%sql
DESCRIBE TABLE EXTENDED gold.dim_customers

In [0]:
df_customers.createOrReplaceTempView("customers")
df_policies.createOrReplaceTempView("policies")

In [0]:
%sql

WITH src AS (
    SELECT DISTINCT
        p.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.phone,
        c.address,
        c.date_of_birth
    FROM policies p
    INNER JOIN customers c
        ON p.customer_id = p.customer_id  -- adjust if needed
)

MERGE INTO gold.dim_customers d
USING src s
ON d.customer_id = s.customer_id

WHEN MATCHED THEN UPDATE SET
    d.customer_id = s.customer_id,
    d.first_name = s.first_name,
    d.last_name = s.last_name,
    d.email = s.email,
    d.phone = s.phone,
    d.address = s.address,
    d.date_of_birth = s.date_of_birth

WHEN NOT MATCHED THEN INSERT (
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    address,
    date_of_birth
)
VALUES (
    s.customer_id,
    s.first_name,
    s.last_name,
    s.email,
    s.phone,
    s.address,
    s.date_of_birth
);

In [0]:
%sql
select * from gold.dim_customers

In [0]:
%sql
DROP TABLE IF EXISTS gold.dim_agents;

In [0]:
%sql
CREATE TABLE gold.dim_agents (
    sk_agent BIGINT GENERATED BY DEFAULT AS IDENTITY,
    agent_id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING
) USING DELTA;


In [0]:
df_agents.createOrReplaceTempView("agents")
df_agent_policies.createOrReplaceTempView("agent_policies")

In [0]:
%sql

WITH src AS (
    SELECT DISTINCT
        ap.agent_id,
        a.first_name,
        a.last_name,
        a.email,
        a.phone
    FROM agent_policies ap
    INNER JOIN agents a
        ON ap.agent_id = ap.agent_id  -- adjust if needed
)

MERGE INTO gold.dim_agents d
USING src s
ON d.agent_id = s.agent_id

WHEN MATCHED THEN UPDATE SET
    d.agent_id = s.agent_id,
    d.first_name = s.first_name,
    d.last_name = s.last_name,
    d.email = s.email,
    d.phone = s.phone

WHEN NOT MATCHED THEN INSERT (
    agent_id,
    first_name,
    last_name,
    email,
    phone
)
VALUES (
    s.agent_id,
    s.first_name,
    s.last_name,
    s.email,
    s.phone
);

In [0]:
%sql
select * from gold.dim_agents;

In [0]:
%sql
DROP TABLE IF EXISTS gold.dim_insurance_types;

In [0]:
%sql
CREATE TABLE gold.dim_insurance_types (
    sk_insurance_type BIGINT GENERATED BY DEFAULT AS IDENTITY,
    insurance_type_id BIGINT,
    type_name STRING,
    description STRING
) USING DELTA;

In [0]:
df_insurance_types.createOrReplaceTempView("insurance_types")
df_policies.createOrReplaceTempView("policies")

In [0]:
%sql
WITH src AS (
    SELECT DISTINCT
        p.insurance_type_id,
        i.type_name,
        i.description
    FROM policies p
    INNER JOIN insurance_types i
        ON p.insurance_type_id = p.insurance_type_id
)

MERGE INTO gold.dim_insurance_types d
USING src s
ON d.insurance_type_id = s.insurance_type_id

WHEN MATCHED THEN UPDATE SET
    d.insurance_type_id = s.insurance_type_id,
    d.type_name = s.type_name,
    d.description = s.description

WHEN NOT MATCHED THEN INSERT (
    insurance_type_id,
    type_name,
    description
)
VALUES (
    s.insurance_type_id,
    s.type_name,
    s.description
);

In [0]:
%sql
select * from gold.dim_insurance_types

In [0]:
%sql
drop table if exists gold.fact_policies

In [0]:
%sql
CREATE TABLE gold.fact_policies (
    policy_number STRING,
    customer_id BIGINT,
    insurance_type_id BIGINT,
    start_date DATE,
    end_date DATE,
    premium_amount DOUBLE,
    policy_status STRING
) USING DELTA;

In [0]:
df_policies.createOrReplaceTempView("policies")

In [0]:
%sql
INSERT INTO gold.fact_policies
SELECT
    policy_number,
    customer_id,
    insurance_type_id,
    start_date,
    end_date,
    premium_amount,
    policy_status
FROM policies;

In [0]:
%sql
select * from gold.fact_policies

In [0]:
%sql
DROP TABLE IF EXISTS gold.fact_claims;

In [0]:
%sql
CREATE TABLE gold.fact_claims (
    policy_id BIGINT,
    claim_date DATE,
    claim_amount DOUBLE,
    claim_status STRING
) USING DELTA;

In [0]:
df_claims.createOrReplaceTempView("claims")

In [0]:
%sql
INSERT INTO gold.fact_claims
SELECT
    policy_id,
    claim_date,
    claim_amount,
    claim_status
FROM claims;

In [0]:
%sql
DROP TABLE IF EXISTS gold.fact_payments;

In [0]:
%sql
CREATE TABLE gold.fact_payments (
    policy_id BIGINT,
    payment_date DATE,
    amount DOUBLE,
    payment_method STRING
) USING DELTA;

In [0]:
df_payments.createOrReplaceTempView("payments")

In [0]:
%sql
INSERT INTO gold.fact_payments
SELECT
    policy_id,
    payment_date,
    amount,
    payment_method
FROM payments;

In [0]:
%sql
SELECT * FROM gold.fact_policies;